<a href="https://colab.research.google.com/github/tonHS/Canadian-Crime-Trends/blob/main/notebooks/Crime_Trends_Analysis_Modified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canadian Crime Trends Analysis (Modified)

## Overview
This notebook analyzes Criminal Code crime trends in Canada using Statistics Canada data.

### Data Sources
- **Table 35-10-0177-01**: Incident-based crime statistics (Police-reported crime)
- **Table 35-10-0062-01**: Police-reported crime for selected offences

### Analysis Outputs
1. **Table PNG**: Top 10 violent Criminal Code violations in 2024 (crime rates) with growth metrics
2. **Table PNG**: Top 10 Criminal Code property violations in 2024 (crime rates) with growth metrics
3. **Table PNG**: Top 10 other Criminal Code violations in 2024 (crime rates) with growth metrics
4. **Table PNG**: Shoplifting INCIDENT COUNTS for 2024 with growth metrics (Criminal Code and Organized Crime)
5. **Line Graphs PNG**: Shoplifting trends over time (3 PNG files)

**Author**: tonHS  
**Date**: 2025-11-23

**Note**: Top 10 tables use crime rates per 100,000 population. Shoplifting analysis uses actual incident counts.

## Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install pandas matplotlib seaborn requests openpyxl -q

import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Create directories
data_dir = Path('data')
outputs_dir = Path('outputs')
figures_dir = Path('figures')

for directory in [data_dir, outputs_dir, figures_dir]:
    directory.mkdir(exist_ok=True)

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Setup complete")

## Step 1: Fetch General Crime Data (Table 35-10-0177-01)

In [ ]:
print("=" * 80)
print("FETCHING GENERAL CRIME DATA (TABLE 35-10-0177-01)")
print("=" * 80)

# Table ID for incident-based crime statistics
TABLE_ID_GENERAL = "35100177"

# Download URL
api_url = f"https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{TABLE_ID_GENERAL}/en"

# Implement a retry mechanism for robustness
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def requests_retry_session(retries=5, backoff_factor=0.3, status_forcelist=(429, 500, 502, 503, 504), session=None):
    session = session or requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(['GET', 'POST'])
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

try:
    print(f"\n📥 Requesting download URL from Statistics Canada API...")
    session = requests_retry_session()
    response = session.get(api_url, timeout=60)

    if not response.ok:
        raise ValueError(f"API request failed: {response.status_code} - {response.reason}")

    zip_url = response.json()['object']
    print(f"✓ Got download URL")

    # Download the ZIP file using the retry session
    print(f"📥 Downloading data...")
    zip_response = session.get(zip_url, timeout=60)

    if not zip_response.ok:
        raise ValueError(f"Data download failed: {zip_response.status_code} - {zip_response.reason}")

    # Extract and load the CSV
    with zipfile.ZipFile(BytesIO(zip_response.content)) as z:
        csv_filename = f"{TABLE_ID_GENERAL}.csv"
        if csv_filename not in z.namelist():
            csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]

        print(f"✓ Extracting: {csv_filename}")
        df_general = pd.read_csv(z.open(csv_filename), low_memory=False)

    print(f"✓ Data loaded successfully: {len(df_general):,} rows, {len(df_general.columns)} columns")

    # Save raw data
    raw_path = data_dir / 'general_crime_raw.csv'
    df_general.to_csv(raw_path, index=False)
    print(f"✓ Saved to: {raw_path}")

    # Display data info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Columns: {', '.join(df_general.columns.tolist())}")
    print(f"   • Shape: {df_general.shape}")

except Exception as e:
    print(f"❌ Error: {e}")
    raise

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

## Step 2: Fetch Organized Crime Data (Table 35-10-0062-01)

In [ ]:
print("=" * 80)
print("FETCHING ORGANIZED CRIME DATA (TABLE 35-10-0062-01)")
print("=" * 80)

# Table ID for organized crime (note: no dashes in the download URL!)
TABLE_ID_ORGANIZED = "3510006201"  # Fixed: was "35100062"

# Use direct CSV download with CORRECT URL format
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID_ORGANIZED}-eng.zip"

try:
    print(f"\n📥 Downloading data from Statistics Canada...")
    print(f"URL: {download_url}")
    
    # Download the data
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    
    # Extract ZIP file
    with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
        csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]
        
        if not csv_files:
            raise ValueError("No CSV file found in downloaded ZIP")
        
        csv_filename = csv_files[0]
        print(f"✓ Downloaded and extracted: {csv_filename}")
        
        with zip_file.open(csv_filename) as csv_file:
            df_organized = pd.read_csv(csv_file, low_memory=False)
    
    print(f"✓ Data loaded successfully: {len(df_organized):,} rows, {len(df_organized.columns)} columns")
    
    # Save raw data
    raw_path = data_dir / 'organized_crime_raw.csv'
    df_organized.to_csv(raw_path, index=False)
    print(f"✓ Saved to: {raw_path}")
    
    # Display data info
    print(f"\n📊 Dataset Overview:")
    print(f"   • Time period: {df_organized['REF_DATE'].min()} to {df_organized['REF_DATE'].max()}")
    print(f"   • Columns: {', '.join(df_organized.columns.tolist())}")
    
    # DIAGNOSTIC: Check for shoplifting or theft data
    violation_cols = [col for col in df_organized.columns if 'violation' in col.lower()]
    if violation_cols:
        violation_col = violation_cols[0]
        print(f"\n🔍 Checking for shoplifting/theft data:")
        
        # Try shoplifting first
        shoplifting_mask = df_organized[violation_col].str.contains('shoplifting', case=False, na=False)
        if shoplifting_mask.any():
            shoplifting_types = df_organized[shoplifting_mask][violation_col].unique()
            print(f"   ✓ Found {len(shoplifting_types)} shoplifting violation(s):")
            for i, viol in enumerate(shoplifting_types, 1):
                print(f"      {i}. {viol}")
        else:
            # Try looking for theft categories
            theft_mask = df_organized[violation_col].str.contains('theft', case=False, na=False)
            if theft_mask.any():
                theft_types = df_organized[theft_mask][violation_col].unique()
                print(f"   ⚠ No 'shoplifting' violations found, but found {len(theft_types)} 'theft' violation(s):")
                for i, viol in enumerate(theft_types, 1):
                    print(f"      {i}. {viol}")
                print(f"   • Note: Shoplifting may be included in broader theft categories.")
            else:
                print(f"   ⚠ WARNING: No 'shoplifting' or 'theft' violations found!")
                print(f"   • This table may only contain aggregated categories.")
                print(f"   • Showing first 15 unique violations:")
                for i, viol in enumerate(df_organized[violation_col].unique()[:15], 1):
                    print(f"      {i}. {viol}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\n⚠ WARNING: Could not fetch organized crime data!")
    print(f"   Creating empty dataframe as fallback...")
    print(f"   Organized crime shoplifting data will not be available in the analysis.")
    # Create empty dataframe with expected structure
    df_organized = pd.DataFrame()

print("\n" + "=" * 80)
print("✓ DATA FETCH COMPLETE")
print("=" * 80)

## Step 3: Process and Clean Data (Criminal Code Only)

In [ ]:
print("=" * 80)
print("PROCESSING AND CLEANING DATA (CRIMINAL CODE ONLY)")
print("=" * 80)

# Process general crime data for RATES (for Top 10 tables)
df_general_rates = df_general.copy()
df_general_rates['Year'] = df_general_rates['REF_DATE'].astype(int)
df_general_rates = df_general_rates[df_general_rates['VALUE'].notna()]

print(f"\n📋 Initial data: {len(df_general_rates):,} rows")

# Find violation column
violation_cols = [col for col in df_general_rates.columns if 'violation' in col.lower()]
if violation_cols:
    violation_col_general = violation_cols[0]
    print(f"✓ Violation column: '{violation_col_general}'")
else:
    raise ValueError("Could not find violation column in general crime data")

# Find statistics column
stats_cols = [col for col in df_general_rates.columns if 'statistic' in col.lower()]
if stats_cols:
    stats_col = stats_cols[0]
    print(f"✓ Statistics column: '{stats_col}'")
    print(f"  • Available statistics types: {df_general_rates[stats_col].unique().tolist()}")
else:
    raise ValueError("Could not find statistics column")

# Find geography column
geo_cols = [col for col in df_general_rates.columns if 'geo' in col.lower()]
if geo_cols:
    geo_col = geo_cols[0]
    print(f"✓ Geography column: '{geo_col}'")
else:
    raise ValueError("Could not find geography column")

# CRITICAL FILTERS FOR RATES:
# 1. Filter for CANADA geography only
df_general_rates = df_general_rates[df_general_rates[geo_col] == 'Canada']
print(f"\n✓ Filtered for Canada: {len(df_general_rates):,} rows")

# 2. Filter for RATE PER 100,000 POPULATION only (for Top 10 tables)
df_general_rates = df_general_rates[
    df_general_rates[stats_col].str.contains('Rate per 100,000 population', case=False, na=False)
]
print(f"✓ Filtered for crime rates: {len(df_general_rates):,} rows")

# 3. Filter for CRIMINAL CODE ONLY - Exclude traffic violations and Federal Statutes
exclude_keywords = [
    'traffic',
    'Criminal Code traffic',
    'Federal statute',
    'Federal Statutes',
    'Other federal statute',
    'Controlled Drugs and Substances Act',
    'Youth Criminal Justice Act',
    'Cannabis Act'
]

# Create exclusion mask
exclude_mask = pd.Series([False] * len(df_general_rates), index=df_general_rates.index)
for keyword in exclude_keywords:
    exclude_mask |= df_general_rates[violation_col_general].str.contains(keyword, case=False, na=False)

df_general_rates = df_general_rates[~exclude_mask]
print(f"✓ Excluded traffic and Federal Statutes: {len(df_general_rates):,} rows")

# Exclude total rows
df_general_rates = df_general_rates[
    ~df_general_rates[violation_col_general].str.contains('Total', case=False, na=False)
]
print(f"✓ Excluded totals: {len(df_general_rates):,} rows")

# PROCESS GENERAL CRIME DATA FOR INCIDENTS (for shoplifting analysis)
df_general_incidents = df_general.copy()
df_general_incidents['Year'] = df_general_incidents['REF_DATE'].astype(int)
df_general_incidents = df_general_incidents[df_general_incidents['VALUE'].notna()]

# Filter for Canada
df_general_incidents = df_general_incidents[df_general_incidents[geo_col] == 'Canada']

# Filter for ACTUAL INCIDENTS (not rates)
df_general_incidents = df_general_incidents[
    df_general_incidents[stats_col].str.contains('Actual incidents', case=False, na=False)
]
print(f"\n✓ Incident data filtered: {len(df_general_incidents):,} rows")

# Apply same exclusions to incident data
exclude_mask_inc = pd.Series([False] * len(df_general_incidents), index=df_general_incidents.index)
for keyword in exclude_keywords:
    exclude_mask_inc |= df_general_incidents[violation_col_general].str.contains(keyword, case=False, na=False)

df_general_incidents = df_general_incidents[~exclude_mask_inc]
df_general_incidents = df_general_incidents[
    ~df_general_incidents[violation_col_general].str.contains('Total', case=False, na=False)
]

print(f"\n📋 Final cleaned data (rates):")
print(f"   • Total rows: {len(df_general_rates):,}")
print(f"   • Unique violations: {df_general_rates[violation_col_general].nunique():,}")
print(f"   • Year range: {df_general_rates['Year'].min()} - {df_general_rates['Year'].max()}")

print(f"\n📋 Final cleaned data (incidents):")
print(f"   • Total rows: {len(df_general_incidents):,}")
print(f"   • Unique violations: {df_general_incidents[violation_col_general].nunique():,}")

# Process organized crime data (if available)
if len(df_organized) > 0:
    print(f"\n📊 Processing organized crime data...")
    df_organized_incidents = df_organized.copy()
    df_organized_incidents['Year'] = df_organized_incidents['REF_DATE'].astype(int)
    df_organized_incidents = df_organized_incidents[df_organized_incidents['VALUE'].notna()]

    # Find violation column for organized crime
    violation_cols_org = [col for col in df_organized_incidents.columns if 'violation' in col.lower()]
    if violation_cols_org:
        violation_col_organized = violation_cols_org[0]
        print(f"✓ Organized crime - Violation column: '{violation_col_organized}'")
    else:
        print(f"⚠ Could not find violation column in organized crime data")
        violation_col_organized = None

    # Find statistics and geography columns for organized crime
    stats_cols_org = [col for col in df_organized_incidents.columns if 'statistic' in col.lower()]
    geo_cols_org = [col for col in df_organized_incidents.columns if 'geo' in col.lower()]

    if stats_cols_org:
        stats_col_org = stats_cols_org[0]
        # Filter for actual incidents (not rates)
        df_organized_incidents = df_organized_incidents[
            df_organized_incidents[stats_col_org].str.contains('Actual incidents', case=False, na=False)
        ]
        print(f"✓ Filtered for actual incidents")

    if geo_cols_org:
        geo_col_org = geo_cols_org[0]
        # Check available geographies
        available_geos = df_organized_incidents[geo_col_org].unique()
        print(f"✓ Available geographies: {', '.join(available_geos[:5])}{'...' if len(available_geos) > 5 else ''}")
        
        # Try to filter for Canada, or sum all regions
        if 'Canada' in available_geos:
            df_organized_incidents = df_organized_incidents[df_organized_incidents[geo_col_org] == 'Canada']
            print(f"✓ Filtered for Canada geography")
        else:
            print(f"⚠ 'Canada' total not found. Will aggregate data from all regions.")
            # Keep all regions - we'll sum them later

    # Exclude traffic and Federal Statutes from organized crime data
    if violation_col_organized:
        exclude_mask_org = pd.Series([False] * len(df_organized_incidents), index=df_organized_incidents.index)
        for keyword in exclude_keywords:
            exclude_mask_org |= df_organized_incidents[violation_col_organized].str.contains(keyword, case=False, na=False)

        df_organized_incidents = df_organized_incidents[~exclude_mask_org]

        # Exclude totals
        df_organized_incidents = df_organized_incidents[
            ~df_organized_incidents[violation_col_organized].str.contains('Total', case=False, na=False)
        ]

    print(f"✓ Organized crime incident data cleaned: {len(df_organized_incidents):,} rows")
    if violation_col_organized:
        print(f"   • Unique violations: {df_organized_incidents[violation_col_organized].nunique():,}")
else:
    print(f"\n⚠ Organized crime data not available - creating empty dataframe")
    df_organized_incidents = pd.DataFrame()
    violation_col_organized = None

print("\n" + "=" * 80)
print("✓ DATA PROCESSING COMPLETE")
print("=" * 80)

## Helper Function: Save Table as PNG

In [ ]:
def save_table_as_png(df, filename, title, color='#2E86AB'):
    """
    Save a pandas DataFrame as a PNG image.
    
    Parameters:
    - df: pandas DataFrame to save
    - filename: output filename (including path)
    - title: title for the table
    - color: header color (hex code)
    """
    fig, ax = plt.subplots(figsize=(14, len(df) * 0.5 + 1))
    ax.axis('tight')
    ax.axis('off')
    
    # Create table
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc='left',
        loc='center',
        colWidths=[0.05] + [0.35] + [0.12] * (len(df.columns) - 2)
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Style header
    for i in range(len(df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(color)
        cell.set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(df) + 1):
        for j in range(len(df.columns)):
            cell = table[(i, j)]
            if i % 2 == 0:
                cell.set_facecolor('#f0f0f0')
            else:
                cell.set_facecolor('white')
    
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.savefig(filename, bbox_inches='tight', dpi=300, facecolor='white')
    plt.close()
    
    print(f"✓ Saved table PNG: {filename}")

print("✓ Helper function defined")

## Step 4: Analyze Top 10 Violent Criminal Code Violations (2024 Rates)

In [ ]:
print("=" * 80)
print("TOP 10 VIOLENT CRIMINAL CODE VIOLATIONS (2024 CRIME RATES)")
print("=" * 80)

# Keywords for identifying violent crimes
violent_keywords = [
    'homicide', 'murder', 'manslaughter', 'assault', 'sexual', 'robbery',
    'kidnapping', 'abduction', 'extortion', 'criminal harassment',
    'uttering threats', 'discharge firearm', 'weapons offence',
    'threatening', 'forcible', 'attempted murder', 'pointing a firearm'
]

# Filter for violent crimes
mask = df_general_rates[violation_col_general].str.lower().str.contains(
    '|'.join(violent_keywords), na=False
)
df_violent = df_general_rates[mask].copy()

print(f"\n✓ Found {len(df_violent):,} records for violent crimes")

# Get 2024 data
df_2024 = df_violent[df_violent['Year'] == 2024]
violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)

print(f"✓ Found {len(violations_2024)} unique violent crime types in 2024")

# Get top 10
top_10_violent = violations_2024.head(10)

# Calculate growth metrics
results = []
for violation in top_10_violent.index:
    rate_2024 = violations_2024.get(violation, 0)
    
    # Get 2019 data (5-year comparison)
    df_2019 = df_violent[df_violent['Year'] == 2019]
    rate_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
    
    # Get 2014 data (10-year comparison)
    df_2014 = df_violent[df_violent['Year'] == 2014]
    rate_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
    
    # Calculate growth rates
    growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
    growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
    
    results.append({
        'Rank': len(results) + 1,
        'Violation': violation,
        '2024 Rate': round(rate_2024, 2),
        '2019 Rate': round(rate_2019, 2) if rate_2019 > 0 else 0,
        '2014 Rate': round(rate_2014, 2) if rate_2014 > 0 else 0,
        'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
        'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
    })

df_violent_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'top_10_violent_crimes_2024.csv'
df_violent_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Save as PNG
png_path = figures_dir / 'top_10_violent_crimes_2024.png'
save_table_as_png(
    df_violent_results,
    png_path,
    'Top 10 Violent Criminal Code Violations (2024) - Crime Rates per 100,000',
    color='#2E86AB'
)

# Display table
print("\n" + "=" * 120)
print("TOP 10 VIOLENT CRIMINAL CODE VIOLATIONS (2024)")
print("=" * 120)
print(df_violent_results.to_string(index=False))
print("=" * 120)

print("\n✓ ANALYSIS COMPLETE")

## Step 5: Analyze Top 10 Property Criminal Code Violations (2024 Rates)

In [ ]:
print("=" * 80)
print("TOP 10 PROPERTY CRIMINAL CODE VIOLATIONS (2024 CRIME RATES)")
print("=" * 80)

# Keywords for identifying property crimes
property_keywords = [
    'theft', 'break and enter', 'breaking and entering', 'fraud', 'mischief', 'arson',
    'shoplifting', 'motor vehicle theft', 'possession of stolen',
    'identity theft', 'identity fraud', 'forgery', 'altering'
]

# Filter for property crimes
mask = df_general_rates[violation_col_general].str.lower().str.contains(
    '|'.join(property_keywords), na=False
)
df_property = df_general_rates[mask].copy()

print(f"\n✓ Found {len(df_property):,} records for property crimes")

# Get 2024 data
df_2024 = df_property[df_property['Year'] == 2024]
violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)

print(f"✓ Found {len(violations_2024)} unique property crime types in 2024")

# Get top 10
top_10_property = violations_2024.head(10)

# Calculate growth metrics
results = []
for violation in top_10_property.index:
    rate_2024 = violations_2024.get(violation, 0)
    
    # Get 2019 data (5-year comparison)
    df_2019 = df_property[df_property['Year'] == 2019]
    rate_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
    
    # Get 2014 data (10-year comparison)
    df_2014 = df_property[df_property['Year'] == 2014]
    rate_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
    
    # Calculate growth rates
    growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
    growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
    
    results.append({
        'Rank': len(results) + 1,
        'Violation': violation,
        '2024 Rate': round(rate_2024, 2),
        '2019 Rate': round(rate_2019, 2) if rate_2019 > 0 else 0,
        '2014 Rate': round(rate_2014, 2) if rate_2014 > 0 else 0,
        'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
        'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
    })

df_property_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'top_10_property_crimes_2024.csv'
df_property_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Save as PNG
png_path = figures_dir / 'top_10_property_crimes_2024.png'
save_table_as_png(
    df_property_results,
    png_path,
    'Top 10 Property Criminal Code Violations (2024) - Crime Rates per 100,000',
    color='#A23B72'
)

# Display table
print("\n" + "=" * 120)
print("TOP 10 PROPERTY CRIMINAL CODE VIOLATIONS (2024)")
print("=" * 120)
print(df_property_results.to_string(index=False))
print("=" * 120)

print("\n✓ ANALYSIS COMPLETE")

## Step 6: Analyze Top 10 Other Criminal Code Violations (2024 Rates)

In [ ]:
print("=" * 80)
print("TOP 10 OTHER CRIMINAL CODE VIOLATIONS (2024 CRIME RATES)")
print("=" * 80)

# Identify crimes that are NOT violent or property crimes
violent_keywords = [
    'homicide', 'murder', 'manslaughter', 'assault', 'sexual', 'robbery',
    'kidnapping', 'abduction', 'extortion', 'criminal harassment',
    'uttering threats', 'discharge firearm', 'weapons offence',
    'threatening', 'forcible', 'attempted murder', 'pointing a firearm'
]

property_keywords = [
    'theft', 'break and enter', 'breaking and entering', 'fraud', 'mischief', 'arson',
    'shoplifting', 'motor vehicle theft', 'possession of stolen',
    'identity theft', 'identity fraud', 'forgery', 'altering'
]

# Create masks for violent and property crimes
violent_mask = df_general_rates[violation_col_general].str.lower().str.contains(
    '|'.join(violent_keywords), na=False
)
property_mask = df_general_rates[violation_col_general].str.lower().str.contains(
    '|'.join(property_keywords), na=False
)

# Filter for "other" crimes (neither violent nor property)
df_other = df_general_rates[~(violent_mask | property_mask)].copy()

print(f"\n✓ Found {len(df_other):,} records for other Criminal Code violations")

# Get 2024 data
df_2024 = df_other[df_other['Year'] == 2024]
violations_2024 = df_2024.groupby(violation_col_general)['VALUE'].sum().sort_values(ascending=False)

print(f"✓ Found {len(violations_2024)} unique other crime types in 2024")

# Get top 10
top_10_other = violations_2024.head(10)

# Calculate growth metrics
results = []
for violation in top_10_other.index:
    rate_2024 = violations_2024.get(violation, 0)
    
    # Get 2019 data (5-year comparison)
    df_2019 = df_other[df_other['Year'] == 2019]
    rate_2019 = df_2019[df_2019[violation_col_general] == violation]['VALUE'].sum()
    
    # Get 2014 data (10-year comparison)
    df_2014 = df_other[df_other['Year'] == 2014]
    rate_2014 = df_2014[df_2014[violation_col_general] == violation]['VALUE'].sum()
    
    # Calculate growth rates
    growth_5yr = ((rate_2024 - rate_2019) / rate_2019 * 100) if rate_2019 > 0 else np.nan
    growth_10yr = ((rate_2024 - rate_2014) / rate_2014 * 100) if rate_2014 > 0 else np.nan
    
    results.append({
        'Rank': len(results) + 1,
        'Violation': violation,
        '2024 Rate': round(rate_2024, 2),
        '2019 Rate': round(rate_2019, 2) if rate_2019 > 0 else 0,
        '2014 Rate': round(rate_2014, 2) if rate_2014 > 0 else 0,
        'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
        'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
    })

df_other_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'top_10_other_crimes_2024.csv'
df_other_results.to_csv(output_path, index=False)
print(f"\n✓ Results saved to: {output_path}")

# Save as PNG
png_path = figures_dir / 'top_10_other_crimes_2024.png'
save_table_as_png(
    df_other_results,
    png_path,
    'Top 10 Other Criminal Code Violations (2024) - Crime Rates per 100,000',
    color='#F18F01'
)

# Display table
print("\n" + "=" * 120)
print("TOP 10 OTHER CRIMINAL CODE VIOLATIONS (2024)")
print("=" * 120)
print(df_other_results.to_string(index=False))
print("=" * 120)

print("\n✓ ANALYSIS COMPLETE")

## Step 7: Analyze Shoplifting INCIDENT COUNTS (2024)

In [ ]:
print("=" * 80)
print("SHOPLIFTING INCIDENT COUNTS ANALYSIS (2024)")
print("=" * 80)

# Define shoplifting categories
shoplifting_categories = {
    'Criminal Code - Shoplifting $5,000 or under': r'Shoplifting.*(\\$5,000|5,000).*under',
    'Criminal Code - Shoplifting over $5,000': r'Shoplifting.*over.*(\\$5,000|5,000)',
}

results = []

# Analyze Criminal Code shoplifting
print("\\n📊 Processing Criminal Code shoplifting incident data...")
for category, pattern in shoplifting_categories.items():
    # Filter for this shoplifting category
    df_category = df_general_incidents[
        df_general_incidents[violation_col_general].str.contains(pattern, case=False, na=False, regex=True)
    ]
    
    if len(df_category) > 0:
        # Get 2024 incidents
        incidents_2024 = df_category[df_category['Year'] == 2024]['VALUE'].sum()
        
        # Get 2019 incidents
        incidents_2019 = df_category[df_category['Year'] == 2019]['VALUE'].sum()
        
        # Get 2014 incidents
        incidents_2014 = df_category[df_category['Year'] == 2014]['VALUE'].sum()
        
        # Calculate growth
        growth_5yr = ((incidents_2024 - incidents_2019) / incidents_2019 * 100) if incidents_2019 > 0 else np.nan
        growth_10yr = ((incidents_2024 - incidents_2014) / incidents_2014 * 100) if incidents_2014 > 0 else np.nan
        
        results.append({
            'Category': category,
            '2024 Incidents': int(incidents_2024),
            '2019 Incidents': int(incidents_2019),
            '2014 Incidents': int(incidents_2014),
            'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
            'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
        })
        print(f"  ✓ {category}: {int(incidents_2024):,} incidents")

# Analyze Organized Crime shoplifting (if data exists and has shoplifting violations)
if len(df_organized_incidents) > 0 and violation_col_organized is not None:
    print("\\n📊 Processing Organized Crime shoplifting incident data...")
    shoplifting_org_categories = {
        'Organized Crime - Shoplifting $5,000 or under': r'Shoplifting.*(\\$5,000|5,000).*under',
        'Organized Crime - Shoplifting over $5,000': r'Shoplifting.*over.*(\\$5,000|5,000)',
    }

    found_org_crime_shoplifting = False
    for category, pattern in shoplifting_org_categories.items():
        # Filter for this shoplifting category
        df_category = df_organized_incidents[
            df_organized_incidents[violation_col_organized].str.contains(pattern, case=False, na=False, regex=True)
        ]
        
        if len(df_category) > 0:
            found_org_crime_shoplifting = True
            # Get 2024 incidents
            incidents_2024 = df_category[df_category['Year'] == 2024]['VALUE'].sum()
            
            # Get 2019 incidents
            incidents_2019 = df_category[df_category['Year'] == 2019]['VALUE'].sum()
            
            # Get 2014 incidents
            incidents_2014 = df_category[df_category['Year'] == 2014]['VALUE'].sum()
            
            # Calculate growth
            growth_5yr = ((incidents_2024 - incidents_2019) / incidents_2019 * 100) if incidents_2019 > 0 else np.nan
            growth_10yr = ((incidents_2024 - incidents_2014) / incidents_2014 * 100) if incidents_2014 > 0 else np.nan
            
            results.append({
                'Category': category,
                '2024 Incidents': int(incidents_2024),
                '2019 Incidents': int(incidents_2019),
                '2014 Incidents': int(incidents_2014),
                'Growth 2024 vs 2019 (%)': round(growth_5yr, 1) if not pd.isna(growth_5yr) else np.nan,
                'Growth 2024 vs 2014 (%)': round(growth_10yr, 1) if not pd.isna(growth_10yr) else np.nan
            })
            print(f"  ✓ {category}: {int(incidents_2024):,} incidents")
    
    if not found_org_crime_shoplifting:
        print("  ⚠ No specific 'shoplifting' violations found in organized crime data.")
        print("  • NOTE: Table 35-10-0062-01 may only contain aggregated categories.")
        print("  • Shoplifting may be included in broader 'Robbery and theft' categories.")
else:
    print("\\n⚠ Organized crime data not available.")
    print("  • Skipping organized crime shoplifting analysis.")

df_shoplifting_results = pd.DataFrame(results)

# Save to CSV
output_path = outputs_dir / 'shoplifting_incidents_2024.csv'
df_shoplifting_results.to_csv(output_path, index=False)
print(f"\\n✓ Results saved to: {output_path}")

# Save as PNG
png_path = figures_dir / 'shoplifting_incidents_2024.png'
save_table_as_png(
    df_shoplifting_results,
    png_path,
    'Shoplifting Incident Counts (2024) - Criminal Code and Organized Crime',
    color='#6A4C93'
)

# Display formatted table
print("\\n" + "=" * 120)
print("SHOPLIFTING INCIDENT COUNTS (2024)")
print("=" * 120)
print(df_shoplifting_results.to_string(index=False))
print("=" * 120)

print("\\n✓ ANALYSIS COMPLETE")

## Step 8: Create Shoplifting Trend Line Graphs

In [ ]:
print("=" * 80)
print("CREATING SHOPLIFTING TREND LINE GRAPHS")
print("=" * 80)

# Graph 1: Criminal Code Shoplifting $5,000 and under
print("\\n📊 Creating graph 1: Criminal Code Shoplifting $5,000 and under...")
pattern_under = r'Shoplifting.*(\\$5,000|5,000).*under'
df_cc_under = df_general_incidents[
    df_general_incidents[violation_col_general].str.contains(pattern_under, case=False, na=False, regex=True)
]
df_cc_under_yearly = df_cc_under.groupby('Year')['VALUE'].sum().reset_index()
df_cc_under_yearly.columns = ['Year', 'Incidents']

plt.figure(figsize=(12, 6))
plt.plot(df_cc_under_yearly['Year'], df_cc_under_yearly['Incidents'], 
         marker='o', linewidth=2.5, markersize=8, color='#2E86AB')
plt.title('Criminal Code Shoplifting $5,000 and Under - Incident Trends', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Incidents', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(df_cc_under_yearly['Year'], rotation=45)
plt.tight_layout()
graph1_path = figures_dir / 'shoplifting_cc_under_5000_trend.png'
plt.savefig(graph1_path, dpi=300, facecolor='white')
plt.close()
print(f"✓ Saved: {graph1_path}")

# Graph 2: Criminal Code Shoplifting over $5,000
print("\\n📊 Creating graph 2: Criminal Code Shoplifting over $5,000...")
pattern_over = r'Shoplifting.*over.*(\\$5,000|5,000)'
df_cc_over = df_general_incidents[
    df_general_incidents[violation_col_general].str.contains(pattern_over, case=False, na=False, regex=True)
]
df_cc_over_yearly = df_cc_over.groupby('Year')['VALUE'].sum().reset_index()
df_cc_over_yearly.columns = ['Year', 'Incidents']

plt.figure(figsize=(12, 6))
plt.plot(df_cc_over_yearly['Year'], df_cc_over_yearly['Incidents'], 
         marker='o', linewidth=2.5, markersize=8, color='#A23B72')
plt.title('Criminal Code Shoplifting Over $5,000 - Incident Trends', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Incidents', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(df_cc_over_yearly['Year'], rotation=45)
plt.tight_layout()
graph2_path = figures_dir / 'shoplifting_cc_over_5000_trend.png'
plt.savefig(graph2_path, dpi=300, facecolor='white')
plt.close()
print(f"✓ Saved: {graph2_path}")

# Graph 3: Organized Crime Shoplifting (both over and under $5,000)
print("\\n📊 Creating graph 3: Organized Crime Shoplifting (over and under $5,000)...")

# Check if organized crime data exists and has shoplifting data
if len(df_organized_incidents) > 0 and violation_col_organized is not None:
    # Get organized crime under $5,000
    df_org_under = df_organized_incidents[
        df_organized_incidents[violation_col_organized].str.contains(pattern_under, case=False, na=False, regex=True)
    ]
    df_org_under_yearly = df_org_under.groupby('Year')['VALUE'].sum().reset_index()
    df_org_under_yearly.columns = ['Year', 'Incidents_Under']

    # Get organized crime over $5,000
    df_org_over = df_organized_incidents[
        df_organized_incidents[violation_col_organized].str.contains(pattern_over, case=False, na=False, regex=True)
    ]
    df_org_over_yearly = df_org_over.groupby('Year')['VALUE'].sum().reset_index()
    df_org_over_yearly.columns = ['Year', 'Incidents_Over']

    # Merge the data
    df_org_combined = pd.merge(df_org_under_yearly, df_org_over_yearly, on='Year', how='outer').fillna(0)

    # Only create graph if we have data
    if len(df_org_combined) > 0 and (df_org_combined['Incidents_Under'].sum() > 0 or df_org_combined['Incidents_Over'].sum() > 0):
        plt.figure(figsize=(12, 6))
        plt.plot(df_org_combined['Year'], df_org_combined['Incidents_Under'], 
                 marker='o', linewidth=2.5, markersize=8, label='Under $5,000', color='#6A4C93')
        plt.plot(df_org_combined['Year'], df_org_combined['Incidents_Over'], 
                 marker='s', linewidth=2.5, markersize=8, label='Over $5,000', color='#F18F01')
        plt.title('Organized Crime Shoplifting - Incident Trends', 
                  fontsize=14, fontweight='bold', pad=20)
        plt.xlabel('Year', fontsize=12, fontweight='bold')
        plt.ylabel('Number of Incidents', fontsize=12, fontweight='bold')
        plt.legend(fontsize=11, loc='best')
        plt.grid(True, alpha=0.3)
        plt.xticks(df_org_combined['Year'], rotation=45)
        plt.tight_layout()
        graph3_path = figures_dir / 'shoplifting_organized_crime_trend.png'
        plt.savefig(graph3_path, dpi=300, facecolor='white')
        plt.close()
        print(f"✓ Saved: {graph3_path}")
    else:
        print("⚠ No organized crime shoplifting data found for graph.")
        print("  • Creating placeholder graph with note...")
        
        # Create a blank graph with a note
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.text(0.5, 0.5, 'Organized Crime Shoplifting Data Not Available\\n\\nTable 35-10-0062-01 may only contain aggregated categories.\\nShoplifting may be included in broader \\'Robbery and theft\\' categories.',
                ha='center', va='center', fontsize=14, wrap=True)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        plt.title('Organized Crime Shoplifting - Incident Trends', 
                  fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()
        graph3_path = figures_dir / 'shoplifting_organized_crime_trend.png'
        plt.savefig(graph3_path, dpi=300, facecolor='white')
        plt.close()
        print(f"✓ Saved placeholder: {graph3_path}")
else:
    print("⚠ Organized crime data not available.")
    print("  • Creating placeholder graph...")
    
    # Create a blank graph with a note
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.text(0.5, 0.5, 'Organized Crime Data Not Available\\n\\nTable 35-10-0062-01 could not be accessed or does not contain\\nshoplifting as a separate violation category.',
            ha='center', va='center', fontsize=14, wrap=True)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    plt.title('Organized Crime Shoplifting - Incident Trends', 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    graph3_path = figures_dir / 'shoplifting_organized_crime_trend.png'
    plt.savefig(graph3_path, dpi=300, facecolor='white')
    plt.close()
    print(f"✓ Saved placeholder: {graph3_path}")

print("\\n" + "=" * 80)
print("✓ ALL GRAPHS CREATED SUCCESSFULLY")
print("=" * 80)
print(f"\\nGraphs saved to:")
print(f"  1. {graph1_path}")
print(f"  2. {graph2_path}")
print(f"  3. {graph3_path}")

## Summary and Conclusion

In [ ]:
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 Outputs Generated:")

print("\n1. Top 10 Violent Criminal Code Violations (2024)")
print(f"   • CSV: {outputs_dir}/top_10_violent_crimes_2024.csv")
print(f"   • PNG: {figures_dir}/top_10_violent_crimes_2024.png")
print(f"   • Metric: Crime rates per 100,000 population")

print("\n2. Top 10 Property Criminal Code Violations (2024)")
print(f"   • CSV: {outputs_dir}/top_10_property_crimes_2024.csv")
print(f"   • PNG: {figures_dir}/top_10_property_crimes_2024.png")
print(f"   • Metric: Crime rates per 100,000 population")

print("\n3. Top 10 Other Criminal Code Violations (2024)")
print(f"   • CSV: {outputs_dir}/top_10_other_crimes_2024.csv")
print(f"   • PNG: {figures_dir}/top_10_other_crimes_2024.png")
print(f"   • Metric: Crime rates per 100,000 population")

print("\n4. Shoplifting Incident Counts (2024)")
print(f"   • CSV: {outputs_dir}/shoplifting_incidents_2024.csv")
print(f"   • PNG: {figures_dir}/shoplifting_incidents_2024.png")
print(f"   • Metric: Actual incident counts (NOT rates)")
print(f"   • Categories: Criminal Code and Organized Crime (over/under $5,000)")
print(f"   • Includes: Growth vs 2019 and 2014")

print("\n5. Shoplifting Trend Line Graphs")
print(f"   • {figures_dir}/shoplifting_cc_under_5000_trend.png")
print(f"   • {figures_dir}/shoplifting_cc_over_5000_trend.png")
print(f"   • {figures_dir}/shoplifting_organized_crime_trend.png")

print("\n" + "=" * 80)
print("DATA QUALITY NOTES")
print("=" * 80)
print("\n✓ All data filtered for Criminal Code violations ONLY")
print("✓ Traffic violations EXCLUDED")
print("✓ Federal Statute violations EXCLUDED")
print("✓ Top 10 tables use CRIME RATES per 100,000 population")
print("✓ Shoplifting analysis uses ACTUAL INCIDENT COUNTS")
print("✓ Data filtered for Canada geography only")
print("✓ All total rows excluded to prevent double-counting")

print("\n" + "=" * 80)
print("✓ ALL ANALYSIS TASKS COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Author: tonHS")